# Notebook 5: P-Values — The Most Misunderstood Concept in Statistics

**Difficulty:** ⭐⭐⭐⭐ | **Estimated Time:** 2.5 hours  
**Theme:** House Price Prediction  
**Week 3 — Statistics for ML & Python Data Stack**

---

## Learning Objectives
By the end of this notebook you will be able to:
- State the **precise, correct** definition of a p-value
- List at least four things a p-value is **not**
- Demonstrate empirically why 5% false-positive rates arise even when H₀ is true
- Apply Bonferroni correction when testing multiple features
- Distinguish **statistical significance** from **practical significance**
- Explain p-value hacking and why it inflates false-positive rates


## 1. Why Does This Matter for Machine Learning?

Imagine you're building a model to predict house prices. You have 200 candidate features — square footage, number of bathrooms, proximity to schools, crime rate, colour of the front door, alignment of the house with cardinal directions, etc.

You test each feature for correlation with price and report all features where `p < 0.05`. A manager glances at the list and says: *"Great, these are definitely real predictors!"*

**But wait** — with 200 features and α = 0.05, you'd expect **10 false positives by pure chance** even if none of those features had any real relationship with price. The front-door colour might look significant. Compass alignment might look significant.

P-values are **everywhere** in ML:
- Feature selection: which features are "significantly" correlated with the target?
- A/B testing: is Model A significantly better than Model B?
- Research papers: almost every ML paper reports p-values to validate claims
- Misusing p-values leads to **overfitting, false discoveries, irreproducible results**

> **The replication crisis in science** — where many published findings couldn't be reproduced — is largely a p-value misuse crisis. Understanding p-values correctly is a superpower in ML.


## 2. The Fire Alarm Analogy

You're the building manager. Someone claims a fire just broke out on the 3rd floor. A fire alarm is blaring.

You ask: **"If there were NO fire, how likely is it that the alarm would be going off this loudly?"**

If the answer is *"very unlikely — maybe 3% of the time the alarm would go off this loudly with no fire"*, that's your p-value: **p = 0.03**.

Notice what the p-value tells you:
- It tells you how surprising your **evidence** is, **assuming nothing is happening**
- It does **NOT** tell you "there's a 3% chance there's no fire"
- It does **NOT** tell you "there's a 97% chance there IS a fire"
- It tells you: *the alarm is ringing in a way that would be unusual if there were no fire*

**In statistics:**
- "No fire" = Null Hypothesis (H₀): there is no real difference or effect
- "Alarm blaring" = your observed data (e.g., the sample mean difference you measured)
- p-value = P(observing data this extreme OR MORE extreme | H₀ is true)

---

### House Price Version

You measure average house prices in District A ($310K) and District B ($290K). You wonder: *"Could this $20K difference just be random sampling noise?"*

H₀: The true mean price is the same in both districts.  
H₁: The true mean prices differ.

p-value = P(seeing a difference of $20K or more between samples | both districts actually have the same true mean price)

If p = 0.03: *"If both districts had identical true prices, we'd still see a $20K gap between samples 3% of the time by chance."*


## 3. Plain English: What IS a P-Value?

A p-value answers one very specific question:

> **"Assuming the null hypothesis is completely true, what is the probability of observing data at least as extreme as what we actually observed?"**

Key words to burn into your memory:
- **"Assuming H₀ is true"** — this is a conditional probability. We start by PRETENDING H₀ is true.
- **"at least as extreme"** — we count our observation AND everything more extreme in the same direction
- **It is a probability about the DATA, not about the hypothesis**

### The Formal Definition

$$p\text{-value} = P(T \geq t_{\text{observed}} \mid H_0 \text{ is true})$$

Where:
- $T$ is the test statistic (e.g., t-statistic, z-score, F-statistic)
- $t_{\text{observed}}$ is the test statistic computed from your actual sample
- The "$\geq$" means "at least as extreme" (for a one-tailed test; use $|T| \geq |t_{\text{obs}}|$ for two-tailed)
- The conditioning on $H_0$ is crucial — this probability is computed **under the null distribution**

### For a Two-Sample t-Test on House Prices

$$t = \frac{\bar{x}_A - \bar{x}_B}{\sqrt{\frac{s_A^2}{n_A} + \frac{s_B^2}{n_B}}}$$

Where:
- $\bar{x}_A, \bar{x}_B$ = sample means of districts A and B
- $s_A^2, s_B^2$ = sample variances
- $n_A, n_B$ = sample sizes

The p-value is then the area in the tails of the t-distribution beyond $|t_{\text{observed}}|$.


## 4. What a P-Value is NOT

This is where most people go wrong. Let's be crystal clear:

| ❌ Wrong interpretation | ✅ Why it's wrong |
|---|---|
| "p = 0.03 means there's a 3% chance H₀ is true" | P-value is P(data \| H₀), NOT P(H₀ \| data). You'd need Bayes' theorem for that. |
| "p = 0.03 means there's a 97% chance H₁ is true" | Same error — this confuses P(data \| hypothesis) with P(hypothesis \| data) |
| "p = 0.03 means the result is due to chance only 3% of the time" | The result either IS or IS NOT due to chance — there's no probability to assign to a single outcome |
| "p < 0.05 means the effect is large or important" | A tiny, irrelevant effect in a huge dataset can have p = 0.000001 |
| "p = 0.06 means there's no effect" | It means the evidence doesn't clear the threshold. Absence of evidence ≠ evidence of absence. |
| "p < 0.05 means the result will replicate" | False — 50% of 'significant' findings fail to replicate |

### The Correct Interpretation Template:

> *"If the null hypothesis were true (no real difference in house prices between districts), the probability of observing a sample difference as large as [X] or larger purely by chance is [p-value]."*

That's it. That's the whole thing. Nothing more, nothing less.


In [ ]:
# ============================================================
# SETUP: Import all required libraries
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats

# Set random seed for reproducibility — same results every run
np.random.seed(42)

# Set a clean visual style for all plots
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')  # colorblind-friendly palette

# Matplotlib display settings for clearer output
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

print("Libraries loaded successfully!")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

## 5. Simulation: What Does p=0.05 Actually Look Like?

The best way to understand p-values is to **simulate** them.

**The experiment setup:**
- We create a world where H₀ is **definitely, 100% true** — both districts have **identical** true mean house prices
- We pretend we don't know this and repeatedly sample from both districts
- We run a t-test each time and record the p-value
- We ask: *how often do we get p < 0.05 even though H₀ is absolutely true?*

The answer should be approximately 5% — that's literally what α = 0.05 means.


In [ ]:
# ============================================================
# SIMULATION: 10,000 experiments where H₀ is TRUE
# Both districts have IDENTICAL true mean prices ($300K)
# ============================================================

# Ground truth parameters — both populations are IDENTICAL
TRUE_MEAN = 300_000      # $300,000 — same for both districts
TRUE_STD  = 50_000       # $50,000 standard deviation — same for both
SAMPLE_SIZE = 30         # 30 houses sampled per district per experiment
N_EXPERIMENTS = 10_000   # run 10,000 independent experiments

# Storage for results
p_values_under_null = []    # p-value from each experiment
t_stats_under_null  = []    # t-statistic from each experiment

# Run the simulation
for _ in range(N_EXPERIMENTS):
    # Sample 30 houses from District A — drawn from N(300K, 50K)
    sample_A = np.random.normal(TRUE_MEAN, TRUE_STD, SAMPLE_SIZE)
    # Sample 30 houses from District B — SAME distribution!
    sample_B = np.random.normal(TRUE_MEAN, TRUE_STD, SAMPLE_SIZE)
    
    # Run independent two-sample t-test (two-tailed)
    # This tests: is the mean of A different from the mean of B?
    t_stat, p_val = stats.ttest_ind(sample_A, sample_B)
    
    p_values_under_null.append(p_val)
    t_stats_under_null.append(t_stat)

# Convert to numpy arrays for easy analysis
p_values_under_null = np.array(p_values_under_null)
t_stats_under_null  = np.array(t_stats_under_null)

# Count how many experiments yielded a "significant" result (p < 0.05)
n_false_positives = np.sum(p_values_under_null < 0.05)
false_positive_rate = n_false_positives / N_EXPERIMENTS

print("=" * 55)
print("SIMULATION RESULTS: H₀ is TRUE in all experiments")
print("=" * 55)
print(f"Total experiments run      : {N_EXPERIMENTS:,}")
print(f"Experiments with p < 0.05  : {n_false_positives:,}")
print(f"False positive rate        : {false_positive_rate:.2%}")
print(f"\nExpected false positive rate: ~5.00%")
print(f"\nConclusion: Even when H₀ is DEFINITELY TRUE, we get")
print(f"'significant' results ~{false_positive_rate:.1%} of the time!")

In [ ]:
# ============================================================
# VISUALIZATION 1: Distribution of p-values when H₀ is TRUE
# Under H₀, p-values follow a UNIFORM distribution on [0, 1]
# This is a mathematical fact, and we verify it empirically here
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# --- Left plot: Histogram of all p-values ---
ax1 = axes[0]
ax1.hist(p_values_under_null, bins=50, color='steelblue', edgecolor='white',
         alpha=0.85, density=True)  # density=True normalises to probability density

# Draw the theoretical uniform distribution as a red horizontal line
ax1.axhline(y=1.0, color='red', linestyle='--', linewidth=2,
            label='Theoretical uniform distribution (y=1)')

# Shade the "false positive" region (p < 0.05) in orange
ax1.axvspan(0, 0.05, alpha=0.3, color='orange',
            label=f'False positive zone (p < 0.05)\n≈{false_positive_rate:.1%} of experiments')

ax1.set_xlabel('p-value')
ax1.set_ylabel('Density')
ax1.set_title('P-value Distribution Under H₀\n(H₀ is TRUE in every single experiment)')
ax1.legend(fontsize=10)
ax1.set_xlim(0, 1)

# Add annotation explaining the key insight
ax1.annotate('Flat / Uniform!\nEvery p-value equally\nlikely when H₀ is true',
             xy=(0.5, 0.5), xytext=(0.55, 0.7),
             fontsize=10, color='darkblue',
             arrowprops=dict(arrowstyle='->', color='darkblue'))

# --- Right plot: Zoomed into p-values below 0.05 ---
ax2 = axes[1]
# Filter to show only the p-values that would be called "significant"
sig_p_values = p_values_under_null[p_values_under_null < 0.05]

ax2.hist(sig_p_values, bins=20, color='orange', edgecolor='white', alpha=0.85)
ax2.set_xlabel('p-value (only showing p < 0.05)')
ax2.set_ylabel('Count')
ax2.set_title(f'"Significant" Results When H₀ is TRUE\n'
              f'({n_false_positives:,} false positives out of {N_EXPERIMENTS:,} tests)')

# Add a text box summarising the key lesson
textstr = (f'These {n_false_positives:,} results would be\n'
            'published as "real discoveries"\n'
            'but H₀ was TRUE every time!')
ax2.text(0.03, 0.78, textstr, transform=ax2.transAxes,
         fontsize=10, verticalalignment='top',
         bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.suptitle('The False Positive Problem: α=0.05 Means 5% False Positives by Design',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\nKey insight: Under H₀, p-values are UNIFORMLY distributed.")
print("This means p=0.03 is just as likely as p=0.63 when H₀ is true.")
print("The 'significance threshold' is a policy choice, not a truth detector.")

## 6. The Multiple Testing Problem

Imagine you're a data scientist at a real estate company. You have a dataset with 50 house features and you want to find which ones correlate with price. You test each feature for correlation and report anything with p < 0.05.

**The math of multiple testing:**

Probability of at least one false positive when running $m$ independent tests, each at level $\alpha$:

$$P(\text{at least one false positive}) = 1 - (1-\alpha)^m$$

With α = 0.05 and m = 50 tests:
$$P(\text{at least one false positive}) = 1 - (0.95)^{50} \approx 0.923$$

**You have a 92.3% chance of finding at least one false positive!** With 50 features, you'd expect 2–3 false discoveries even if none of them are actually related to price.

### Bonferroni Correction

The simplest fix: divide your significance threshold by the number of tests.

$$\alpha_{\text{Bonferroni}} = \frac{\alpha}{m} = \frac{0.05}{50} = 0.001$$

This keeps the **family-wise error rate** (FWER) at α = 0.05 across all tests.


In [ ]:
# ============================================================
# DEMONSTRATION: Multiple Testing Problem with House Features
# We create 50 fake features — NONE of which are truly related
# to house prices. We then test each one and see how many look
# 'significant' by pure chance.
# ============================================================

np.random.seed(7)   # different seed for this section

N_HOUSES  = 100     # number of houses in our dataset
N_FEATURES = 50     # number of candidate features to test
ALPHA = 0.05        # standard significance threshold

# Generate house prices — drawn from a normal distribution
house_prices = np.random.normal(300_000, 50_000, N_HOUSES)

# Generate 50 RANDOM features — no relationship with prices whatsoever
# Each feature is pure noise drawn from a standard normal distribution
fake_features = np.random.normal(0, 1, (N_HOUSES, N_FEATURES))

# Test each feature's correlation with price using Pearson correlation
feature_results = []

for i in range(N_FEATURES):
    feature_col = fake_features[:, i]   # extract the i-th feature column
    # Pearson r and its two-tailed p-value
    r_stat, p_val = stats.pearsonr(feature_col, house_prices)
    feature_results.append({
        'feature': f'Feature_{i+1:02d}',
        'r': r_stat,
        'p_value': p_val,
        'significant_uncorrected': p_val < ALPHA,
        'significant_bonferroni': p_val < (ALPHA / N_FEATURES)  # Bonferroni corrected
    })

# Build a DataFrame for easy analysis
results_df = pd.DataFrame(feature_results).sort_values('p_value')

# Count how many features pass each threshold
n_sig_uncorrected = results_df['significant_uncorrected'].sum()
n_sig_bonferroni  = results_df['significant_bonferroni'].sum()

bonferroni_alpha = ALPHA / N_FEATURES

print("=" * 60)
print("MULTIPLE TESTING: 50 RANDOM (NULL) FEATURES vs House Prices")
print("=" * 60)
print(f"None of the {N_FEATURES} features are actually related to price.")
print()
print(f"Standard threshold (α = {ALPHA})")
print(f"  → 'Significant' features found: {n_sig_uncorrected}")
print(f"  → Expected by chance: {N_FEATURES * ALPHA:.1f}")
print()
print(f"Bonferroni correction (α = {ALPHA}/{N_FEATURES} = {bonferroni_alpha:.4f})")
print(f"  → 'Significant' features found: {n_sig_bonferroni}")
print()
print("Top 10 features by p-value (the most 'significant' null features):")
print(results_df[['feature', 'r', 'p_value',
                   'significant_uncorrected', 'significant_bonferroni']].head(10).to_string(index=False))

In [ ]:
# ============================================================
# VISUALIZATION 2: Multiple Testing — Before and After Correction
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Sort features by p-value for better visual display
sorted_results = results_df.reset_index(drop=True)
sorted_results['feature_num'] = range(1, len(sorted_results) + 1)

# Color coding: red = passes threshold, blue = does not
colors_uncorr = ['#e74c3c' if sig else '#3498db'
                 for sig in sorted_results['significant_uncorrected']]
colors_bonf   = ['#e74c3c' if sig else '#3498db'
                 for sig in sorted_results['significant_bonferroni']]

# --- Left: Without Bonferroni correction ---
ax1 = axes[0]
ax1.scatter(sorted_results['feature_num'], sorted_results['p_value'],
            c=colors_uncorr, s=80, zorder=5, edgecolors='white', linewidth=0.5)

# Draw the threshold line
ax1.axhline(ALPHA, color='red', linestyle='--', linewidth=2,
            label=f'α = {ALPHA} threshold')

# Count and annotate false positives
ax1.set_title(f'Without Bonferroni Correction\n'
              f'{n_sig_uncorrected} "significant" features (ALL are false positives!)',
              color='darkred' if n_sig_uncorrected > 0 else 'darkgreen')
ax1.set_xlabel('Feature (sorted by p-value, smallest first)')
ax1.set_ylabel('p-value')
ax1.set_ylim(0, 1)
ax1.legend()

# Add shading for the significant region
ax1.axhspan(0, ALPHA, alpha=0.1, color='red', label='"Significant" zone')

# Add annotation
ax1.text(0.97, 0.95,
         f'{n_sig_uncorrected} false positives\nwould be reported\nas real discoveries!',
         transform=ax1.transAxes, ha='right', va='top', fontsize=10,
         bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))

# --- Right: With Bonferroni correction ---
ax2 = axes[1]
ax2.scatter(sorted_results['feature_num'], sorted_results['p_value'],
            c=colors_bonf, s=80, zorder=5, edgecolors='white', linewidth=0.5)

# Draw the corrected threshold
ax2.axhline(ALPHA, color='red', linestyle=':', linewidth=1.5, alpha=0.5,
            label=f'Original α = {ALPHA}')
ax2.axhline(bonferroni_alpha, color='darkgreen', linestyle='--', linewidth=2,
            label=f'Bonferroni α = {bonferroni_alpha:.4f}')

ax2.set_title(f'With Bonferroni Correction\n'
              f'{n_sig_bonferroni} "significant" features survive correction',
              color='darkgreen')
ax2.set_xlabel('Feature (sorted by p-value, smallest first)')
ax2.set_ylabel('p-value')
ax2.set_ylim(0, 1)
ax2.legend()

# Shading for the Bonferroni zone
ax2.axhspan(0, bonferroni_alpha, alpha=0.1, color='green')

ax2.text(0.97, 0.95,
         f'{n_sig_bonferroni} features survive\nBonferroni correction\n(correct — none are real!)',
         transform=ax2.transAxes, ha='right', va='top', fontsize=10,
         bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.9))

# Add legend for dot colours
red_patch  = mpatches.Patch(color='#e74c3c', label='Passes threshold (false positive)')
blue_patch = mpatches.Patch(color='#3498db', label='Does not pass threshold')
axes[0].legend(handles=[red_patch, blue_patch], loc='upper left', fontsize=9)

plt.suptitle('Multiple Testing Problem: 50 Null Features vs House Prices',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Practical vs Statistical Significance

Here is one of the most important ideas in applied statistics:

> **Statistical significance tells you whether an effect is real (non-zero). It says nothing about whether the effect is meaningful.**

With a large enough sample, **everything** becomes statistically significant — even effects so tiny they're completely useless in practice.

### House Price Example

Suppose you're testing whether houses with north-facing gardens are priced differently from those with south-facing gardens. With:
- **n = 100 houses:** Mean difference = $5,000. p = 0.43. "Not significant."
- **n = 100,000 houses:** Mean difference = $200. p = 0.0001. "Highly significant!"

But wait — is a $200 difference in a house costing $300,000 (0.07% of price) actually useful for your model? **Absolutely not.** Yet the test is screaming "significant!"

This is why **effect size** (covered in Notebook 6) is essential. P-values alone are not enough.


In [ ]:
# ============================================================
# DEMONSTRATION: Same effect, different sample sizes
# Shows how a TINY difference becomes "highly significant"
# with a large enough sample — even when practically useless
# ============================================================

np.random.seed(99)

# The 'true' effect: a $200 difference in house prices
# This is 0.067% of a $300K house — completely negligible
TRUE_EFFECT = 200          # $200 actual price difference
MEAN_PRICE  = 300_000      # $300,000 baseline
PRICE_STD   = 50_000       # $50,000 typical standard deviation

# Test the same effect at multiple sample sizes
sample_sizes = [30, 100, 500, 1_000, 5_000, 10_000, 50_000, 100_000]

results_practical = []
for n in sample_sizes:
    # Simulate District A: mean = $300,000
    sample_A = np.random.normal(MEAN_PRICE, PRICE_STD, n)
    # Simulate District B: mean = $300,200 (only $200 more!)
    sample_B = np.random.normal(MEAN_PRICE + TRUE_EFFECT, PRICE_STD, n)

    # Run t-test
    t_stat, p_val = stats.ttest_ind(sample_A, sample_B)

    # Cohen's d: standardised effect size (preview of Notebook 6)
    # d = (mean difference) / (pooled standard deviation)
    pooled_std = np.sqrt((np.var(sample_A, ddof=1) + np.var(sample_B, ddof=1)) / 2)
    cohens_d = abs(np.mean(sample_A) - np.mean(sample_B)) / pooled_std

    results_practical.append({
        'n_per_group': n,
        'mean_A': np.mean(sample_A),
        'mean_B': np.mean(sample_B),
        'observed_diff': np.mean(sample_B) - np.mean(sample_A),
        'p_value': p_val,
        'cohens_d': cohens_d,
        'significant': 'YES ✓' if p_val < 0.05 else 'no'
    })

# Build results table
practical_df = pd.DataFrame(results_practical)

print("=" * 75)
print(f"TRUE EFFECT: ${TRUE_EFFECT} difference (0.067% of ${MEAN_PRICE:,})")
print("=" * 75)
print("\nHow sample size affects statistical significance:")
print()

# Format for display
display_df = practical_df[['n_per_group', 'observed_diff', 'p_value',
                             'cohens_d', 'significant']].copy()
display_df.columns = ['n per group', 'Observed diff ($)', 'p-value', "Cohen's d", 'Significant?']
display_df['Observed diff ($)'] = display_df['Observed diff ($)'].map('${:,.0f}'.format)
display_df['p-value'] = display_df['p-value'].map('{:.6f}'.format)
display_df["Cohen's d"] = display_df["Cohen's d"].map('{:.4f}'.format)

print(display_df.to_string(index=False))
print()
print(f"Cohen's d = {TRUE_EFFECT / PRICE_STD:.4f} — this is TINY (threshold for 'small' is 0.2)")
print("A $200 difference in a $300K house is statistically significant at large n,")
print("but practically USELESS for pricing decisions.")

In [ ]:
# ============================================================
# VISUALIZATION 3: How p-value changes with sample size
# for a FIXED tiny effect (the $200 difference)
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

log_n = np.log10(practical_df['n_per_group'])  # log scale for x-axis readability

# --- Left: p-value vs sample size ---
ax1 = axes[0]
ax1.plot(log_n, practical_df['p_value'], 'o-',
         color='steelblue', linewidth=2.5, markersize=9)

# Draw significance threshold
ax1.axhline(0.05, color='red', linestyle='--', linewidth=2,
            label='α = 0.05 threshold')

# Shade the significant region
ax1.fill_between(log_n, 0, 0.05, alpha=0.1, color='red')

# Mark the crossover point where it becomes significant
sig_mask = practical_df['p_value'] < 0.05
if sig_mask.any():
    first_sig_n = practical_df.loc[sig_mask, 'n_per_group'].iloc[0]
    ax1.axvline(np.log10(first_sig_n), color='orange', linestyle=':',
                linewidth=2, label=f'First "significant" at n={first_sig_n:,}')

# Custom x-tick labels to show actual sample sizes
ax1.set_xticks(log_n)
ax1.set_xticklabels([f'{n:,}' for n in practical_df['n_per_group']], rotation=45)
ax1.set_xlabel('Sample Size (n per group)')
ax1.set_ylabel('p-value')
ax1.set_title(f'P-value vs Sample Size\nTrue effect = ${TRUE_EFFECT} (negligible!)')
ax1.legend()

# --- Right: Cohen's d vs sample size ---
ax2 = axes[1]
ax2.plot(log_n, practical_df['cohens_d'], 'o-',
         color='purple', linewidth=2.5, markersize=9)

# Draw Cohen's d benchmarks
ax2.axhline(0.2, color='green', linestyle='--', linewidth=1.5, alpha=0.8,
            label='d=0.2: Small effect')
ax2.axhline(0.5, color='orange', linestyle='--', linewidth=1.5, alpha=0.8,
            label='d=0.5: Medium effect')
ax2.axhline(0.8, color='red', linestyle='--', linewidth=1.5, alpha=0.8,
            label='d=0.8: Large effect')

ax2.set_xticks(log_n)
ax2.set_xticklabels([f'{n:,}' for n in practical_df['n_per_group']], rotation=45)
ax2.set_xlabel('Sample Size (n per group)')
ax2.set_ylabel("Cohen's d")
ax2.set_title("Effect Size (Cohen's d) vs Sample Size\n"
              "Effect size stays tiny regardless of n")
ax2.legend(fontsize=9)
ax2.set_ylim(0, 0.3)

plt.suptitle('The Practical vs Statistical Significance Paradox\n'
             'p-value → 0 as n → ∞, but Cohen\'s d stays near 0.004',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print("KEY LESSON: p-value depends on BOTH effect size AND sample size.")
print("With enough data, even a $200 difference (irrelevant for $300K homes) looks 'significant'.")
print("Always report effect sizes (Cohen's d) alongside p-values!")

## 8. P-Value Hacking (Data Dredging)

**P-hacking** is the practice of trying many different analyses, subsets, or transformations of data until you get p < 0.05, then reporting only that analysis as if it were your original plan.

Common p-hacking techniques (intentional or not):
- Removing "outliers" until p < 0.05
- Testing multiple subgroups until one is significant
- Adding or removing covariates until p < 0.05
- Collecting more data until p crosses 0.05
- Trying multiple statistical tests and reporting the one that works

Each decision point is another "test", and each test adds to your family-wise error rate.

### The Fix: Pre-registration
- Decide your analysis plan BEFORE looking at the data
- Commit to: specific test, sample size, stopping rule, and covariates
- Report ALL tests run, not just significant ones
- Use hold-out validation sets in ML to prevent data leakage


In [ ]:
# ============================================================
# DEMONSTRATION: P-Value Hacking
# We simulate a scenario where an analyst tries 20 different
# subgroup analyses on the SAME null data, hoping to find
# something 'significant'. Each attempt is a roll of the dice.
# ============================================================

np.random.seed(55)

# Create a large house price dataset with no real group differences
N_TOTAL = 1000     # 1000 houses total
N_GROUPS = 20      # 20 different ways to split the data (suburbs, year built, etc.)

# Baseline prices — all from the same distribution (H₀ is true globally)
all_prices = np.random.normal(300_000, 50_000, N_TOTAL)

# Simulate 20 different subgroup splits (all random noise)
hacking_results = []

for attempt in range(N_GROUPS):
    # Randomly split houses into two 'subgroups'
    # This is like: try removing different "outlier" sets
    # or try different suburb definitions until one sticks
    indices_A = np.random.choice(N_TOTAL, size=50, replace=False)
    remaining  = np.setdiff1d(np.arange(N_TOTAL), indices_A)
    indices_B  = np.random.choice(remaining, size=50, replace=False)

    group_A = all_prices[indices_A]
    group_B = all_prices[indices_B]

    # Test if these randomly defined groups differ
    _, p_val = stats.ttest_ind(group_A, group_B)

    hacking_results.append({
        'attempt': attempt + 1,
        'mean_A': np.mean(group_A),
        'mean_B': np.mean(group_B),
        'p_value': p_val,
        'significant': p_val < 0.05
    })

hacking_df = pd.DataFrame(hacking_results)
n_hacking_sig = hacking_df['significant'].sum()

print("=" * 65)
print("P-VALUE HACKING SIMULATION")
print("All 20 subgroup splits are RANDOM — no real group difference")
print("=" * 65)
print()
print(f"Number of 'significant' results found (p < 0.05): {n_hacking_sig}")
print(f"The 'best' p-value found: {hacking_df['p_value'].min():.4f}")
print()

# Show what a dishonest analyst might report
best_row = hacking_df.loc[hacking_df['p_value'].idxmin()]
print("What the dishonest analyst would publish:")
print(f"  'We found a significant price difference between subgroup A")
print(f"   (mean = ${best_row['mean_A']:,.0f}) and subgroup B")
print(f"   (mean = ${best_row['mean_B']:,.0f}), p = {best_row['p_value']:.4f}'")
print()
print("What they WOULDN'T publish: the other 19 failed attempts.")
print()

# Show all results
print("All 20 attempts:")
print(hacking_df[['attempt', 'mean_A', 'mean_B', 'p_value', 'significant']].to_string(index=False))

In [ ]:
# ============================================================
# VISUALIZATION 4: P-value hacking — showing all 20 attempts
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# --- Left: All 20 p-values from hacking attempts ---
ax1 = axes[0]
colors_hack = ['#e74c3c' if sig else '#95a5a6'
               for sig in hacking_df['significant']]

bars = ax1.bar(hacking_df['attempt'], hacking_df['p_value'],
               color=colors_hack, edgecolor='white', linewidth=0.5)

ax1.axhline(0.05, color='red', linestyle='--', linewidth=2,
            label='α = 0.05 threshold')

ax1.set_xlabel('Subgroup Analysis Attempt Number')
ax1.set_ylabel('p-value')
ax1.set_title(f'P-Hacking: {N_GROUPS} Attempts on Null Data\n'
              f'{n_hacking_sig} appear "significant" by chance')
ax1.legend()
ax1.set_ylim(0, 1)

# Annotate the "published" result
best_attempt = hacking_df.loc[hacking_df['p_value'].idxmin(), 'attempt']
best_pval = hacking_df['p_value'].min()
ax1.annotate(f'This one gets\npublished!\np={best_pval:.3f}',
             xy=(best_attempt, best_pval),
             xytext=(best_attempt + 2, best_pval + 0.1),
             fontsize=10, color='darkred',
             arrowprops=dict(arrowstyle='->', color='darkred'))

# Add legend for colours
red_patch  = mpatches.Patch(color='#e74c3c', label='p < 0.05 ("significant")')
grey_patch = mpatches.Patch(color='#95a5a6', label='p ≥ 0.05 (not significant)')
ax1.legend(handles=[red_patch, grey_patch,
                    plt.Line2D([0], [0], color='red', linestyle='--', linewidth=2,
                               label='α = 0.05')])

# --- Right: What publication bias looks like ---
ax2 = axes[1]

# Simulate what the literature looks like when only p<0.05 is published
# Run 200 experiments, 190 null, 10 where H₁ is true
np.random.seed(123)
null_pvals    = np.random.uniform(0, 1, 190)          # 190 null experiments
true_pvals    = np.random.beta(0.5, 10, 10)            # 10 experiments with real effects

# What actually gets submitted/published (selection bias!)
published_pvals = np.concatenate([null_pvals[null_pvals < 0.05],
                                   true_pvals[true_pvals < 0.05]])
all_pvals_run   = np.concatenate([null_pvals, true_pvals])

ax2.hist(all_pvals_run, bins=20, alpha=0.5, color='steelblue',
         label='All experiments run', density=True)
ax2.hist(published_pvals, bins=10, alpha=0.8, color='orange',
         label='Published (p < 0.05 only)', density=True)
ax2.axhline(1.0, color='red', linestyle=':', linewidth=1.5,
            label='Expected (uniform) under H₀')

ax2.set_xlabel('p-value')
ax2.set_ylabel('Density')
ax2.set_title('Publication Bias\nOnly "significant" studies get published')
ax2.legend(fontsize=9)

plt.suptitle('P-Hacking and Publication Bias — Sources of False Discoveries',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. The Right Way to Use P-Values

P-values are not inherently bad — they are misused. Here are best practices:

### Do:
- **Pre-register** your hypothesis and analysis plan before seeing the data
- **Report effect sizes** (Cohen's d, R², etc.) alongside every p-value
- **Correct for multiple comparisons** when testing many features
- **Report confidence intervals** — they show magnitude, not just direction
- **Distinguish** "statistically significant" from "practically important"
- **Use p-values as one piece of evidence**, not as a binary pass/fail oracle

### Don't:
- Don't treat p = 0.05 as a magic threshold separating truth from fiction
- Don't try multiple analyses and report only the significant one
- Don't interpret p = 0.051 as "no effect" and p = 0.049 as "definite effect"
- Don't forget to consider **prior probability** of the hypothesis being true
- Don't let p-values replace scientific/domain reasoning

### In ML Specifically:
| Context | Better alternative to raw p-value |
|---------|-----------------------------------|
| Feature selection | Use regularization (Lasso) + cross-validation |
| Model comparison | Report RMSE/AUC improvement + Cohen's d on fold performance |
| A/B test | Report lift (%), confidence intervals, business impact |
| Research claim | Report effect size, CI, hold-out set performance |


In [ ]:
# ============================================================
# SUMMARY VISUALIZATION: The Full P-Value Story
# A comprehensive chart showing correct vs incorrect interpretations
# and the p-value curve as effect size changes
# ============================================================

np.random.seed(42)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# --- Left: How p-value changes as true effect grows ---
ax1 = axes[0]

# Range of true effect sizes to test (difference in mean prices)
true_diffs = np.linspace(0, 60_000, 200)   # $0 to $60,000 difference
n_sim = 30                                  # 30 houses per group
n_reps = 500                                # average over 500 repetitions per effect

median_pvals = []
for diff in true_diffs:
    pvals_rep = []
    for _ in range(n_reps):
        a = np.random.normal(MEAN_PRICE, PRICE_STD, n_sim)
        b = np.random.normal(MEAN_PRICE + diff, PRICE_STD, n_sim)
        _, pv = stats.ttest_ind(a, b)
        pvals_rep.append(pv)
    # Use median p-value across repetitions (more stable than mean)
    median_pvals.append(np.median(pvals_rep))

median_pvals = np.array(median_pvals)

ax1.plot(true_diffs / 1000, median_pvals, color='steelblue', linewidth=2.5)
ax1.axhline(0.05, color='red', linestyle='--', linewidth=2, label='α = 0.05')
ax1.fill_between(true_diffs / 1000, 0, 0.05, alpha=0.1, color='red',
                 label='"Significant" zone')

# Mark the crossover — where does the effect become detectable?
crossover_idx = np.argmin(np.abs(median_pvals - 0.05))
crossover_diff = true_diffs[crossover_idx]
ax1.axvline(crossover_diff / 1000, color='orange', linestyle=':', linewidth=2,
            label=f'Detectable at ~${crossover_diff/1000:.0f}K difference (n={n_sim})')

ax1.set_xlabel('True Mean Difference ($K, thousands)')
ax1.set_ylabel('Median p-value (over 500 simulations)')
ax1.set_title(f'P-Value Curve: How p changes as true effect grows\n'
              f'(n={n_sim} per group, σ=${PRICE_STD/1000:.0f}K)')
ax1.legend(fontsize=9)
ax1.set_ylim(0, 1)

# --- Right: Summary table as visual ---
ax2 = axes[1]
ax2.axis('off')  # turn off axes — we'll draw a table

# Create a summary table of correct vs incorrect interpretations
rows = [
    ['CORRECT', 'Prob of seeing this data or more extreme | H₀ is true'],
    ['WRONG #1', 'Prob that H₀ is true'],
    ['WRONG #2', 'Prob that H₁ is true'],
    ['WRONG #3', 'Prob the result is due to chance'],
    ['WRONG #4', 'A measure of effect size / importance'],
    ['WRONG #5', 'Proof that result will replicate'],
]
col_labels = ['Status', 'Interpretation of p-value']

table = ax2.table(cellText=rows, colLabels=col_labels,
                  loc='center', cellLoc='left')
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.5, 2.5)

# Colour the cells
table[0, 0].set_facecolor('#2c3e50')  # header
table[0, 0].set_text_props(color='white', fontweight='bold')
table[0, 1].set_facecolor('#2c3e50')
table[0, 1].set_text_props(color='white', fontweight='bold')

table[1, 0].set_facecolor('#27ae60')  # correct = green
table[1, 0].set_text_props(color='white', fontweight='bold')
table[1, 1].set_facecolor('#d5f5e3')

for row in range(2, 7):              # wrong interpretations = red
    table[row, 0].set_facecolor('#e74c3c')
    table[row, 0].set_text_props(color='white', fontweight='bold')
    table[row, 1].set_facecolor('#fdedec')

ax2.set_title('What a p-value IS vs IS NOT', fontsize=13, fontweight='bold', pad=20)

plt.suptitle('P-Value Summary: The Full Picture', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 10. Self-Check Questions

Test your understanding. Try to answer each question before reading the answer.

---

### Question 1
You compare house prices in two districts and get p = 0.04. Your colleague says: *"There's a 4% chance this result is due to chance."* How do you correct them?

<details>
<summary>Click to reveal answer</summary>

**Your colleague is wrong.** The correct interpretation is:

*"If the two districts had truly identical mean prices (H₀ is true), we would observe a sample difference as large as the one we measured about 4% of the time, purely due to random sampling."*

What p = 0.04 is NOT saying:
- It is NOT saying there's a 4% chance H₀ is true
- It is NOT saying there's a 96% chance the districts are different
- The probability is about the **data given the hypothesis**, not the **hypothesis given the data**

To compute "probability H₀ is true", you'd need Bayes' theorem and a prior probability — p-values alone cannot give you that.
</details>

---

### Question 2
You test 20 house features for correlation with price. Three features have p < 0.05. Should you trust these findings? What correction should you apply?

<details>
<summary>Click to reveal answer</summary>

**Be skeptical.** With 20 tests at α = 0.05, you'd expect **1 false positive** by chance alone. Having 3 significant results might sound impressive, but 1 of those 3 could easily be a false positive.

Expected false positives = 20 × 0.05 = **1.0**

Apply **Bonferroni correction**: new threshold = 0.05 / 20 = **0.0025**

Re-check: which of the 3 "significant" features have p < 0.0025? Only those should be trusted as real findings (at the 5% family-wise error rate).

Even better: use domain knowledge + cross-validation to validate that the feature genuinely improves model performance.
</details>

---

### Question 3
With n = 1,000,000 houses, you find the mean price in District A is $10 more than District B (p = 0.0001). Is this finding practically useful for a house pricing model?

<details>
<summary>Click to reveal answer</summary>

**No — it is not practically useful**, despite being highly statistically significant.

A $10 difference in a $300,000 house is **0.003%** of the price. No buyer makes decisions based on $10. No model needs to account for $10 differences.

The p = 0.0001 simply reflects having an enormous sample size — even a random instrument error of $10 would be "significant" at n = 1,000,000.

**What to report instead:** Cohen's d (which would be approximately 0.0002 — absurdly small) and a confidence interval like [$8, $12]. This makes the trivial magnitude immediately obvious.

**Lesson:** Always ask "Is this effect large enough to matter in the real world?" separately from "Is this effect statistically distinguishable from zero?"
</details>

---

### Question 4
A researcher runs a study and gets p = 0.052. Another researcher runs the same study and gets p = 0.048. Should these two results lead to completely different conclusions?

<details>
<summary>Click to reveal answer</summary>

**No — absolutely not.** A threshold of p = 0.05 is a **convention**, not a law of nature. The difference between p = 0.052 and p = 0.048 is negligible noise — both studies found similarly weak evidence.

The practice of treating p = 0.049 as "proof" and p = 0.051 as "no evidence" leads to:
- "Cliff effects" in perception of evidence
- P-hacking (collecting just enough data to cross the threshold)
- Misleading meta-analyses

**Better approach:** Report the exact p-value, the effect size, the confidence interval, and let the reader assess the weight of evidence on a continuum.
</details>

---

### Question 5
After reading this notebook, a data scientist says: "P-values are useless — I'll just stop reporting them." Is this a good response?

<details>
<summary>Click to reveal answer</summary>

**No — that overcorrects.** P-values have genuine value when used correctly:

- They quantify evidence against a null hypothesis
- They provide a standardised, comparable metric across studies
- They help distinguish signal from sampling noise

The right response is to **use p-values as one piece of evidence** among several:
1. Report the p-value (correct interpretation)
2. Report the effect size (Cohen's d, R², etc.)
3. Report the confidence interval
4. Apply corrections for multiple testing
5. Use domain knowledge to assess practical importance
6. Validate on hold-out data (in ML)

As the statistician Andrew Gelman says: *"The problem is not p-values. The problem is treating p < 0.05 as a license to make strong claims."*
</details>


## Summary

| Concept | Key Takeaway |
|---------|-------------|
| **P-value definition** | P(data this extreme or more \| H₀ is true). About data, not hypotheses. |
| **False positive rate** | At α=0.05, 5% of experiments will appear significant by chance |
| **Under H₀** | P-values are uniformly distributed — any value equally likely |
| **Multiple testing** | k tests → expected k×α false positives; apply Bonferroni: α/k |
| **Sample size effect** | Larger n → smaller p-value for same effect; does NOT mean more important |
| **P-hacking** | Trying many analyses inflates false positives; pre-register to prevent |
| **Practical significance** | A statistically significant effect can still be practically useless |
| **Best practice** | Always report p-value + effect size + confidence interval together |

---

**Next:** Notebook 6 — Confidence Intervals & Cohen's d: measuring HOW MUCH, not just whether.
